In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import napari
import numpy as np
import seaborn as sns
from glasbey import create_palette
from matplotlib import rcParams

palette = {
    "green": "#558150",
    "beige": "#F1E2C3",
    "brown": "#A7785A",
    "pink": "#F0D6C2",
    "black": "#0E0E0E",
}

rcParams["font.family"] = "sans-serif"
rcParams["figure.facecolor"] = "#FFFFFF00"
rcParams["axes.facecolor"] = "#FFFFFF00"
rcParams["legend.framealpha"] = 0.2
rcParams["axes.edgecolor"] = palette["black"]
rcParams["axes.labelcolor"] = palette["black"]
rcParams["xtick.color"] = palette["black"]
rcParams["ytick.color"] = palette["black"]
rcParams["text.color"] = palette["black"]
rcParams["axes.titlecolor"] = palette["black"]

s_palette = sns.cubehelix_palette(as_cmap=True)
pal = sns.color_palette("dark")
cpal = sns.cubehelix_palette(start=-0.25, rot=2, as_cmap=True)
g_palette = create_palette(2000)

In [ ]:
def plot_napari(viewer: napari.Viewer, df, column):
    df["reflection_z"] = 2.1 * df["POSITION_Z"].max() - df["POSITION_Z"]
    df["reflection_x"] = df["POSITION_X"].max() - df["POSITION_X"]

    df["is_reflected"] = df["POSITION_X"] > df["POSITION_X"].max() / 2

    df["display_x"] = df["reflection_x"] * df["is_reflected"] + df["POSITION_X"] * (
        ~df["is_reflected"]
    )
    df["display_z"] = df["reflection_z"] * df["is_reflected"] + df["POSITION_Z"] * (
        ~df["is_reflected"]
    )

    track_id = np.nan_to_num(df[column].unique())
    print(len(track_id))
    color_map = {track: g_palette[i % 2000] for i, track in enumerate(track_id)}
    color = [color_map[track] for track in df[column].fillna(0)]

    viewer.add_points(
        df[["FRAME", "display_x", "POSITION_Y", "display_z"]],
        name=column,
        properties={"spot id": df.index, column: df[column]},
        face_color=color,
        size=13,
    )

In [ ]:
from lxml import etree

from nucleitracking.models.new_tracking import process_trackmate_tree

source_path = Path(r"/data/interim/lightsheet/2025_02_06")
tracked_points_path = source_path / "trackedspots2.xml"
save_path = source_path / "hybrid3"
save_path.mkdir(exist_ok=True)

tree = etree.parse(str(tracked_points_path))
initial_spots_df, graph = process_trackmate_tree(tree)

plt.hist(initial_spots_df.groupby("linear_track_id")["FRAME"].min(), bins=100)
print(f"final frame: {initial_spots_df['FRAME'].max()}")
plt.show()

In [ ]:
# import napari
#
# viewer = napari.Viewer()
# plot_napari(viewer, initial_spots_df, "linear_track_id")
# viewer.theme = 'dark'
# napari.run()

In [ ]:
from nucleitracking.models.new_tracking import interpolate_points

interpolated_spots_df, interpolated_graph = interpolate_points(initial_spots_df, graph)

In [ ]:
from nucleitracking.models.new_tracking import merge_close_tracklets

print(len(interpolated_spots_df.index), len(interpolated_graph.nodes))
merged_spots_df, merged_graph = merge_close_tracklets(
    interpolated_spots_df, interpolated_graph, max_dis=12
)
print(len(merged_spots_df.index), len(merged_graph.nodes))

In [ ]:
from nucleitracking.models.new_tracking import map_divisions

interphase_dividers = [45, 80, 130, 195, 267]

mapped_graph, test_spots_df = map_divisions(
    merged_spots_df, merged_graph, interphase_dividers, new_track_cost=25
)
# mapped_graph, test_spots_df = map_divisions(interpolated_spots_df, interpolated_graph, interphase_dividers, new_track_cost=25)

In [ ]:
from nucleitracking.models.new_tracking import process_graph

spots_df = process_graph(merged_spots_df, mapped_graph)
# spots_df = process_graph(interpolated_spots_df, mapped_graph)
print(
    spots_df[spots_df["FRAME"] == spots_df["FRAME"].max()]
    .groupby("track_id")
    .size()
    .value_counts()
)
spots_df.to_csv(save_path / "spots.csv")

In [ ]:
spots_df

In [ ]:
# 42072, 42074

# print(spots_df.loc[42078, ["POSITION_X", "POSITION_Y", "POSITION_Z"]])
# print(spots_df.loc[42076, ["POSITION_X", "POSITION_Y", "POSITION_Z"]])
# print(spots_df.loc[39164, ["POSITION_X", "POSITION_Y", "POSITION_Z"]])
# print(spots_df.loc[39162, ["POSITION_X", "POSITION_Y", "POSITION_Z"]])

In [ ]:
best_spots = spots_df[spots_df["track_id"] > 0].copy()
n_tracklets = best_spots["track_id"].map(
    best_spots.groupby("track_id")["tracklet_id"].nunique()
)
best_spots = best_spots[n_tracklets > 1]

In [ ]:
import napari
import napari_animation
import PIL as P

viewer = napari.Viewer(ndisplay=3)
# plot_napari(viewer, interpolated_spots_df, "interpolated")
# plot_napari(viewer, interpolated_spots_df, "linear_track_id")
# plot_napari(viewer, spots_df, "is_swapped")
# plot_napari(viewer, spots_df, "linear_track_id")
plot_napari(viewer, best_spots, "track_id")
plot_napari(viewer, test_spots_df, "status")
viewer.theme = "dark"
# viewer._canvas_size = (1800, 1800)


napari.run()

history of waves what has been observed
need to look at it individually over the entire embryo
how many observed
check the properties


In [ ]:
import imageio
from PIL import Image


def make_gif_loop(input_path, output_path):
    """
    Edits the GIF header to loop indefinitely.

    Args:
    input_path (str): Path to the input GIF file.
    output_path (str): Path to save the modified GIF file.
    """
    # try:
    # Open the GIF file using PIL
    with Image.open(input_path) as img:
        # Get the frames and duration
        frames = []
        for i in range(1, img.n_frames):
            img.seek(i)
            frames.append(np.array(img.copy()))
            duration = img.info["duration"]

        # Save the GIF with loop=0 (infinite loop)
        imageio.mimsave(output_path, frames, duration=duration / 1000, loop=0)
        print(f"GIF saved to {output_path} with infinite loop.")

In [ ]:
make_gif_loop("tmp_out.gif", "out.gif")

In [ ]:
with P.Image.open("out.gif") as image:
    print(image.info)
    print(image.__dict__)

In [ ]:
import networkx as nx

df = interpolated_spots_df
gp = interpolated_graph
# tracklet = np.random.choice(df["linear_track_id"].unique())
tracklet = 1014
pts = df[df["linear_track_id"] == tracklet]["graph_key"].values
graph_test = gp.subgraph(pts)

print(df.loc[pts, "FRAME"].min(), df.loc[pts, "FRAME"].max())

nx.draw(
    graph_test,
    with_labels=True,
    pos={
        k: (v["FRAME"] // 10, v["FRAME"] % 10)
        for k, v in df[df["graph_key"].isin(graph_test.nodes())].iterrows()
    },
)
plt.show()